# 07 — RQ1, tầng 2: BUG / FEATURE / UX

**Phạm vi**

Tinh chỉnh hai chiến lược có huấn luyện cho tầng 2 là ViSoBERT và Bi-LSTM + fastText. Tầng 2 chỉ làm việc trên review actionable.

**Dữ liệu huấn luyện và đánh giá**

Huấn luyện trên các review BUG, FEATURE và UX của Train, bỏ các review được đánh dấu ambiguous vì protocol không dùng chúng để huấn
luyện tầng 2. Khi đánh giá, các review ambiguous của Validation được giữ, vì mỗi review vẫn có đúng một nhãn gold.

**Đánh giá trên gold-actionable**

Bước này đánh giá trên nhãn gold-actionable, tức là tách riêng khỏi lỗi của tầng 1; việc ghép hai tầng được làm ở Giai đoạn 10.

**Quy trình**

Giống tầng 1: dò một lưới hyperparameter nhỏ với một seed, chọn cấu hình có Macro-F1 3 lớp cao nhất trên phần actionable của
Validation, rồi chạy lại cấu hình thắng với ba seed (42, 1337, 2024). Tầng 2 không có ngưỡng: lớp dự đoán là lớp có xác suất cao nhất.

**Phân bố Validation**

Validation đã được cân lại ở amendment v1.8 để có 22 review FEATURE, nên phân bố lớp actionable trên Validation không còn là phân bố tự
nhiên.

**Số trên Validation lạc quan**

Cùng một Validation vừa dùng để chọn cấu hình vừa để báo kết quả, nên các con số ở đây sẽ lạc quan hơn thực tế.

**Focused set để dành cho Giai đoạn 10**

Focused FEATURE/UX set (56 review FEATURE) là tập đủ ca để đo ranh giới FEATURE/UX, nhưng theo amendment v1.7 nó bị khóa như Hold-out
và chỉ được mở một lần sau khi mọi tuning đã chốt, nên không được đánh giá ở đây. Mô hình của ba seed ở cấu hình thắng cho mỗi chiến
lược được lưu lại, để Giai đoạn 10 đánh giá focused set và Hold-out một lần bằng đúng các mô hình đã báo cáo trên Validation. Hold-out
và focused set không có mặt trên Kaggle ở notebook này.

**Lấy mã nguồn**

Giống các notebook trước, mã nguồn được lấy từ repository riêng tư bằng token trong Kaggle Secrets, và token được gỡ khỏi cấu hình git
ngay sau khi clone.

In [ ]:
import os, subprocess, sys, json
from kaggle_secrets import UserSecretsClient
REPO_URL = 'github.com/dnhphuong-05/kltn-vn-app-review-requirements.git'
REPO_DIR = '/kaggle/working/repo'
token = UserSecretsClient().get_secret('GITHUB_TOKEN')
if not os.path.exists(REPO_DIR):
    r = subprocess.run(['git', 'clone', '-q', f'https://{token}@{REPO_URL}', REPO_DIR], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError('git clone thất bại: ' + r.stderr.replace(token, '***'))
subprocess.run(['git', '-C', REPO_DIR, 'remote', 'set-url', 'origin', f'https://{REPO_URL}'], check=True)
del token
sys.path.append(os.path.join(REPO_DIR, '02_src'))
import yaml, transformers, sentencepiece
from src.utils.seeds import set_seed
set_seed(42)
CFG = yaml.safe_load(open(os.path.join(REPO_DIR, '03_configs', 'frozen', 'classification.yaml'), encoding='utf-8'))
print('transformers', transformers.__version__, '| sentencepiece', sentencepiece.__version__)
print('commit:', subprocess.run(['git', '-C', REPO_DIR, 'log', '-1', '--format=%h %s'], capture_output=True, text=True).stdout.strip())

## Dữ liệu tầng 2

In [ ]:
import pandas as pd, numpy as np, torch
from src.classification.tier2_bfu.mapping import to_tier2, TIER2_LABELS

def find_data_dir(root='/kaggle/input', target='shopee_train.csv'):
    for dirpath, _, files in os.walk(root):
        if target in files:
            return dirpath
    found = [os.path.join(d, f) for d, _, fs in os.walk(root) for f in fs]
    raise FileNotFoundError(f'Không tìm thấy {target} trong {root}. Các file hiện có:\n' + ('\n'.join(found) or '(trống)'))

DATA_DIR = find_data_dir()
train_df = pd.read_csv(os.path.join(DATA_DIR, 'shopee_train.csv'), encoding='utf-8-sig')
val_df = pd.read_csv(os.path.join(DATA_DIR, 'shopee_validation.csv'), encoding='utf-8-sig')
assert not set(train_df['review_id']) & set(val_df['review_id']), 'Train và Validation trùng review'
T2_TRAIN = to_tier2(train_df, drop_ambiguous=True)          # huấn luyện: bỏ ambiguous
T2_VAL = to_tier2(val_df, drop_ambiguous=False)              # đánh giá: giữ ambiguous
n_amb = int((to_tier2(train_df, drop_ambiguous=False)['ambiguous_flag'].astype(str) == '1').sum())
print('Train tầng 2:', len(T2_TRAIN), T2_TRAIN['gold_label'].value_counts().reindex(TIER2_LABELS).to_dict(), f'(đã bỏ {n_amb} ambiguous)')
print('Validation tầng 2:', len(T2_VAL), T2_VAL['gold_label'].value_counts().reindex(TIER2_LABELS).to_dict(),
      '| ambiguous giữ lại:', int((T2_VAL['ambiguous_flag'].astype(str) == '1').sum()))
print('Các file trên Kaggle:', sorted(os.listdir(DATA_DIR)))
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'không có')
from src.utils.kaggle_paths import working_dir
OUT = working_dir('rq1_tier2')
CKPT = os.path.join(OUT, 'checkpoints')
SEEDS = CFG['learning_curve_seeds']

## ViSoBERT: dò hyperparameter

**Lưới**

Learning rate {1e-5, 2e-5, 3e-5} như tầng 1, nhưng số epoch tăng lên {4, 6, 8}.

**Lý do tăng số epoch**

Train tầng 2 chỉ khoảng 320 review, tức khoảng 20 bước cập nhật mỗi epoch, ít hơn nhiều so với tầng 1.

**Cấu hình và tokenizer**

Các tham số khác giữ như cấu hình đóng băng; tokenizer là bản sentencepiece đã sửa ở Giai đoạn 6 (hàm huấn luyện tự kiểm tra
`self_check` trước khi chạy).

**Cách chọn**

Mỗi cấu hình chạy một seed và được chọn theo Macro-F1 3 lớp trên Validation.

In [ ]:
from src.classification.rq1 import (VisobertRunner, BilstmRunner, grid_search, run_seeds, predictions_frame,
                                    cost_row, save_dev_config)
from src.classification.tier2_bfu.evaluate import argmax_metrics, summary_row, per_class_table, confusion

VIS_GRID = [{'lr': lr, 'epochs': ep} for lr in (1e-5, 2e-5, 3e-5) for ep in (4, 6, 8)]
vis = VisobertRunner(CFG['strategy_b_visobert'])
vis_grid, vis_best = grid_search(vis, VIS_GRID, T2_TRAIN, T2_VAL, 3, TIER2_LABELS, argmax_metrics,
                                 select='macro_f1', seed=42, strategy='visobert')
vis_grid.to_csv(os.path.join(OUT, 'tier2_visobert_grid.csv'), index=False)
print(vis_grid[['config', 'macro_f1', 'f1_BUG', 'f1_FEATURE', 'f1_UX', 'train_seconds']].round(4).to_string(index=False))
print('Cấu hình thắng:', vis_best)

## ViSoBERT: ba seed

**Lưu checkpoint**

Cấu hình thắng được chạy lại với ba seed; mô hình của từng seed được lưu vào `checkpoints/visobert/`, kèm file sentencepiece để nạp lại
đúng tokenizer.

In [ ]:
vis_runs = run_seeds(vis, vis_best, SEEDS, T2_TRAIN, T2_VAL, 3, TIER2_LABELS, strategy='visobert',
                     save_dir=os.path.join(CKPT, 'visobert'))
predictions_frame(T2_VAL, vis_runs, TIER2_LABELS, 'visobert', 'tier2').to_csv(
    os.path.join(OUT, 'tier2_visobert_val_predictions.csv'), index=False)

## Bi-LSTM + fastText: dò hyperparameter và ba seed

**Embedding**

Vector fastText tiếng Việt cố định như các bước trước.

**Lưới**

Kích thước lớp ẩn {64, 128, 256}, dropout {0,3; 0,5}, learning rate {1e-3; 2e-3} và số epoch {10, 20}.

**Lý do thêm số epoch**

Cùng lý do với ViSoBERT: dữ liệu tầng 2 nhỏ nên mỗi epoch có rất ít bước cập nhật. Cách chọn cấu hình và chạy ba seed giống hệt
ViSoBERT.

In [ ]:
from src.classification.strategy_bilstm_fasttext.embeddings import build_vocab, download, load_vectors
BCFG = CFG['strategy_a_bilstm']
bil_runs = None
try:
    vec = download(BCFG['fasttext_url'], '/tmp/cc.vi.300.vec.gz')
    vocab = build_vocab(pd.concat([train_df['text'], val_df['text']]).astype(str))
    emb, coverage = load_vectors(vec, vocab, dim=int(BCFG['embedding_dim']))
    print(f'Từ vựng {len(vocab):,} | có vector fastText: {coverage:.1%}')
    bil = BilstmRunner(BCFG, vocab, emb)
    BIL_GRID = [{'hidden_size': h, 'dropout': d, 'lr': lr, 'epochs': ep}
                for h in (64, 128, 256) for d in (0.3, 0.5) for lr in (1e-3, 2e-3) for ep in (10, 20)]
    bil_grid, bil_best = grid_search(bil, BIL_GRID, T2_TRAIN, T2_VAL, 3, TIER2_LABELS, argmax_metrics,
                                     select='macro_f1', seed=42, strategy='bilstm')
    bil_grid.to_csv(os.path.join(OUT, 'tier2_bilstm_grid.csv'), index=False)
    print(bil_grid[['config', 'macro_f1', 'f1_BUG', 'f1_FEATURE', 'f1_UX', 'train_seconds']].round(4).head(8).to_string(index=False))
    print('Cấu hình thắng:', bil_best)
    bil_runs = run_seeds(bil, bil_best, SEEDS, T2_TRAIN, T2_VAL, 3, TIER2_LABELS, strategy='bilstm',
                         save_dir=os.path.join(CKPT, 'bilstm'))
    predictions_frame(T2_VAL, bil_runs, TIER2_LABELS, 'bilstm', 'tier2').to_csv(
        os.path.join(OUT, 'tier2_bilstm_val_predictions.csv'), index=False)
except Exception as e:
    print('Chưa chạy được Bi-LSTM ở notebook này. Lý do:', repr(e))

## Kiểm tra checkpoint

**Mục đích**

Giai đoạn 10 đánh giá Hold-out và focused set bằng các mô hình đã lưu, nên mô hình seed 42 của mỗi chiến lược được nạp lại, dự đoán lại
Validation và so với xác suất đã có. Nếu khớp thì các checkpoint dùng được cho Giai đoạn 10.

In [ ]:
import gc
pairs = [('visobert', VisobertRunner(CFG['strategy_b_visobert']), vis_runs)]
if bil_runs:
    pairs.append(('bilstm', BilstmRunner(BCFG, {}, None), bil_runs))
for name, runner, runs in pairs:
    model = runner.load(runs[0]['checkpoint'])
    probs, _ = runner.predict_proba(model, T2_VAL['text'].astype(str).tolist())
    diff = float(np.abs(probs - runs[0]['probs']).max())
    print(f'{name}: nạp lại {runs[0]["checkpoint"]} | chênh xác suất lớn nhất {diff:.2e}')
    assert diff < 1e-3, f'{name}: checkpoint không tái tạo được dự đoán'
    del model; gc.collect(); torch.cuda.empty_cache()

## Kết quả tầng 2 trên Validation-actionable

**Bảng chính**

Macro-F1 và F1 từng lớp, trung bình và độ lệch chuẩn qua ba seed.

**Precision và recall từng lớp**

Bảng thứ hai tách precision và recall của từng lớp để thấy mô hình nhầm theo hướng nào.

**Ma trận nhầm lẫn**

Ma trận nhầm lẫn của lần chạy seed 42 cho biết các ca sai rơi vào cặp lớp nào, nhất là ranh giới FEATURE/UX.

**Độ nhạy theo số ca**

Validation chỉ có 22–27 review mỗi lớp, nên mỗi review đoán sai làm F1 của lớp đó thay đổi vài điểm.

**Chi phí và cấu hình dev**

Chi phí huấn luyện, dự đoán của mỗi mô hình, và file cấu hình dev ghi cấu hình thắng cùng đường dẫn checkpoint.

In [ ]:
y = T2_VAL['y'].to_numpy()
models = [('ViSoBERT', 'visobert', vis_runs, vis_best)]
if bil_runs:
    models.append(('Bi-LSTM + fastText', 'bilstm', bil_runs, bil_best))
perf = pd.DataFrame([summary_row(y, runs, name, best, 'Validation-actionable') for name, _, runs, best in models])
per_class = pd.concat([per_class_table(y, runs, name, 'Validation-actionable') for name, _, runs, _ in models])
costs = pd.DataFrame([cost_row(runs, name) for name, _, runs, _ in models])
perf.to_csv(os.path.join(OUT, 'tier2_validation_results.csv'), index=False)
per_class.to_csv(os.path.join(OUT, 'tier2_validation_per_class.csv'), index=False)
costs.to_csv(os.path.join(OUT, 'tier2_costs.csv'), index=False)
print('Tầng 2 trên Validation-actionable (%, trung bình ± độ lệch chuẩn, 3 seed):')
print(perf.drop(columns=['tập']).to_string(index=False))
print('\nPrecision / recall / F1 từng lớp:')
print(per_class.drop(columns=['tập']).to_string(index=False))
for name, key, runs, _ in models:
    cm = confusion(y, runs[0]['probs'])
    cm.to_csv(os.path.join(OUT, f'tier2_{key}_confusion_seed42.csv'))
    print(f'\nMa trận nhầm lẫn {name} (seed 42):')
    print(cm.to_string())
print('\nChi phí:')
print(costs.round(3).to_string(index=False))

dev = {'tier2': {'rule': {'select': 'macro_f1 (3 lớp) trên Validation-actionable, argmax', 'train': 'actionable, bỏ ambiguous',
                          'eval': 'Validation-actionable, giữ ambiguous', 'seeds': SEEDS}}}
for name, key, runs, best in models:
    dev['tier2'][key] = {'overrides': best, 'checkpoints': [os.path.relpath(r['checkpoint'], OUT).replace(os.sep, '/') for r in runs]}
save_dev_config(os.path.join(OUT, 'tier2_dev_config.yaml'), dev,
                'Tier 2 — cấu hình thắng trên Validation-actionable (Macro-F1 3 lớp). DEV, chưa frozen.\n'
                'Checkpoint 3 seed để GĐ10 đánh giá Hold-out + focused set (khóa) một lần.')
print('\nĐã lưu:', sorted(os.listdir(OUT)))

## Nhận xét

**Cấu hình thắng**

ViSoBERT: learning rate 3e-5 với 8 epoch (Macro-F1 0,763 ở lần dò với seed 42); hai cấu hình đứng đầu đều dùng 8 epoch. Bi-LSTM: lớp ẩn
128, dropout 0,5, learning rate 2e-3 và 20 epoch (0,757).

**Cấu hình ở mép lưới — giới hạn được ghi nhận**

Cả hai cấu hình thắng đều nằm ở mép lưới: số epoch lớn nhất và learning rate lớn nhất đã thử. Lưới không được mở rộng: mỗi lần mở rộng
là thêm một lần chọn cấu hình dựa trên Validation, mà Validation chỉ có 71 review actionable, nên con số tốt lên có thể chỉ là khớp với
chính tập này chứ không phản ánh mô hình tốt hơn thật. Mục tiêu của RQ1 là so sánh các chiến lược theo cùng một quy trình (dò lưới với
một seed, chọn theo Macro-F1, chạy lại ba seed), không phải đẩy F1 của từng mô hình lên cao nhất; kết quả có thể cải thiện thêm một
chút với lưới rộng hơn.

**Hai lưới khác cỡ**

Lưới của Bi-LSTM có 24 cấu hình, nhiều hơn 9 cấu hình của ViSoBERT vì Bi-LSTM huấn luyện rẻ hơn nhiều; nếu có lạc quan do dò nhiều thì
phần đó nghiêng về Bi-LSTM.

**Kết quả ba seed**

ViSoBERT đạt Macro-F1 75,6 ± 2,2 và Bi-LSTM đạt 72,0 ± 3,2. ViSoBERT cao hơn ở cả ba seed, nhưng mức chênh trung bình 3,6 điểm chỉ cỡ
một độ lệch chuẩn.

**Seed dùng để chọn cấu hình**

Với Bi-LSTM, lần chạy seed 42 dùng để chọn cấu hình đạt 0,757, còn hai seed còn lại chỉ đạt 0,695 và 0,709, nghĩa là cấu hình này được
chọn một phần nhờ một lần chạy thuận lợi; vì vậy con số được báo cáo là trung bình ba seed chứ không phải con số lúc dò lưới. Với
ViSoBERT, seed 42 nằm giữa hai seed còn lại (0,731 và 0,774) nên hiện tượng này không rõ.

**Từng lớp — khác biệt nằm ở FEATURE**

Hai mô hình gần như ngang nhau ở BUG (F1 74,2 so với 72,3) và UX (79,5 so với 80,3). Ở FEATURE: 73,1 ± 2,7 với ViSoBERT và 63,5 ± 5,7
với Bi-LSTM. Precision FEATURE của hai mô hình tương đương (79,1 và 76,6), còn recall chênh nhau nhiều (68,2 so với 54,5), tức là
Bi-LSTM bỏ sót nhiều review FEATURE hơn.

**Độ tin cậy của chênh lệch FEATURE**

Validation chỉ có 22 review FEATURE, mỗi review đoán sai làm recall FEATURE thay đổi khoảng 4,5 điểm, nên đây là xu hướng cần kiểm lại
trên focused set ở Giai đoạn 10 chứ chưa phải kết luận.

**Ranh giới khó nhất là BUG/FEATURE**

Ma trận nhầm lẫn cộng dồn ba seed cho thấy lỗi nhiều nhất của cả hai mô hình là đoán review FEATURE thành BUG: 15 trên 66 lượt với
ViSoBERT và 23 trên 66 lượt với Bi-LSTM. Tính cả hai chiều, cặp BUG/FEATURE bị nhầm 24 lượt với ViSoBERT và 29 lượt với Bi-LSTM, cặp
BUG/UX 19 và 17 lượt, còn cặp FEATURE/UX chỉ 9 và 12 lượt.

**Liên hệ với quy tắc R6 và focused set**

BUG/FEATURE chính là ranh giới mà quy tắc R6 trong guideline phải tách: "app làm sai điều lẽ ra làm được" là BUG, còn "app không làm
được điều tôi muốn" là FEATURE. Focused set được thiết kế theo các cụm từ gợi ý FEATURE/UX với giả định đó là ranh giới khó nhất; trên
Validation thì ranh giới khó nhất lại là BUG/FEATURE. Focused set vẫn cần để đo FEATURE vì Validation có quá ít ca, nhưng phân tích
Hold-out sẽ xem kỹ cả cặp BUG/FEATURE.

**So với learning curve**

So với learning curve ở mốc 1.200 (cấu hình mặc định 4 epoch, lr 2e-5, cùng dữ liệu Train tầng 2), ViSoBERT sau khi tune tăng Macro-F1
từ 0,735 lên 0,756. Phần tăng đến gần như hoàn toàn từ FEATURE (0,639 lên 0,731), còn BUG giữ nguyên (0,743 và 0,742) và UX giảm nhẹ
(0,822 xuống 0,795). Vì cùng một Validation được dùng để chọn cấu hình, một phần mức tăng này là lạc quan.

**Chi phí**

Mỗi lần huấn luyện ViSoBERT trên 321 review mất khoảng 18,4 giây, dự đoán 1.000 review mất 1,50 giây và dùng tối đa 2,5 GB bộ nhớ GPU;
Bi-LSTM chỉ mất 1,2 giây huấn luyện, 0,06 giây cho 1.000 review và 0,2 GB. ViSoBERT tốn thời gian huấn luyện gấp khoảng 15 lần và thời
gian dự đoán gấp khoảng 25 lần, nhưng ở quy mô này cả hai đều rất nhỏ: toàn bộ lưới 9 cấu hình ViSoBERT mất khoảng 2 phút GPU, lưới 24
cấu hình Bi-LSTM khoảng 30 giây, chưa tính thời gian tải mô hình và file vector fastText.

**Chưa kết luận**

Chưa kết luận mô hình nào tốt hơn ở tầng 2. So sánh cuối cùng của RQ1 dựa trên Hold-out, có thêm focused set cho lớp FEATURE, và phải
tính cả chi phí của LLM few-shot ở Giai đoạn 8.

**File kết quả và checkpoint**

Các file kết quả trong `/kaggle/working/rq1_tier2/` (trừ thư mục `checkpoints/`) đã được tải về `05_experiments/rq1_classification/`,
và `tier2_dev_config.yaml` đã được chép vào `03_configs/dev/`. Thư mục `checkpoints/` khá nặng nên không tải về máy mà được giữ trong
output của phiên bản notebook này, để gắn làm input cho notebook Giai đoạn 10.